# SQL Worksheet — Week1

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Student Attendance Profile
Build the query in these steps:

**1.1 — Select student columns**
From `dim_student`, select `student_id` and `student_name`.

**1.2 — Add attendance data**
Join `fact_attendance` with a `LEFT JOIN` using `student_key`, then add the attendance record identifier.

**1.3 — Add class and batch data**
Join `dim_class` using `class_key` and `dim_batch` using `batch_key`. Add the batch name.

**1.4 — Group and aggregate**
Group by student and batch name. Count attendance records and count missing phone numbers using `NULLIF` and `CASE`.

**1.5 — Sort the result**
Order by attendance records descending, then student name.

Return one row per student and batch, including students with no attendance records.

In [0]:
--Write your code here
select 
    ds.student_id, ds.student_name, 
    coalesce(db.batch_name,'no batch'),
    count(fa.attendance_id) AS count_of_attendance,
    sum(case when nullif(ds.phone_no,'') is null then 1 else 0 end) AS missing_phoneno

from rivadataplatform.dataproduct.dim_student AS ds 

left join rivadataplatform.dataproduct.fact_attendance AS fa
    on fa.student_key = ds.student_key
left join rivadataplatform.dataproduct.dim_class AS dc
    on dc.class_key = fa.class_key
left join rivadataplatform.dataproduct.dim_batch AS db
    on db.batch_key = fa.batch_key

group by ds.student_id,ds.student_name,db.batch_name
order by count_of_attendance desc, ds.student_name

In [0]:
--Write your code here


## Question 2 — Missing Profile Data by City
Build the query in these steps:

**2.1 — Select student location**
From `dim_student`, select the city and replace null city values with `Unknown city`.

**2.2 — Join attendance**
Join `fact_attendance` to the students using `student_key`, then add the attendance record identifier.

**2.3 — Add class information**
Join `dim_class` using `class_key` and add the class topic, replacing null topics with `Topic not assigned`.

**2.4 — Group and aggregate**
Group by the null-safe city and topic. Count distinct students, attendance records, and missing phone values.

**2.5 — Filter and sort**
Keep only groups with at least one missing phone number. Order by missing phone count descending, then city and topic.

In [0]:
--Write your code here

select 
    coalesce(ds.city,'unknown') as city,
    coalesce(dc.topic,'Topic not assigned') as topic,
    count(distinct student_id) as distinct_students,
    count(distinct attendance_id) as distinct_attendance,
    sum(case when nullif(ds.phone_no,'') is null then 1 else 0 end) as missing_phoneno

from rivadataplatform.dataproduct.dim_student as ds 
left join rivadataplatform.dataproduct.fact_attendance as fa
    on ds.student_key = fa.student_key
left join rivadataplatform.dataproduct.dim_class as dc
    on dc.class_key = fa.class_key

group by city,topic
having missing_phoneno > 0
order by missing_phoneno desc, city, topic


In [0]:
--Write your code here

## Question 3 — Distinct Class Calendar
Build the query in these steps:

**3.1 — Select class date columns**
From `dim_class`, select the class date and class key.

**3.2 — Join attendance**
Join `fact_attendance` using `class_key`, then add the attendance record identifier and `date_key`.

**3.3 — Add batch and calendar details**
Join `dim_batch` using the class batch relationship and join `dim_date` using `date_key`. Add batch name, topic, and day name.

**3.4 — Group and count**
Return one row per distinct class date and count its attendance records.

**3.5 — Sort chronologically**
Order the result by class date.

In [0]:
--Write your code here
select
    dc.class_date,
    dc.class_key,
    count(fa.attendance_id) over (partition by dc.class_date) as attendance_count, 
    fa.date_key

from rivadataplatform.dataproduct.dim_class as dc
left join rivadataplatform.dataproduct.fact_attendance as fa
    on dc.class_key = fa.class_key
left join rivadataplatform.dataproduct.dim_batch as db
    on dc.batch_id = db.batch_id
left join rivadataplatform.dataproduct.dim_date as dd
    on fa.date_key = dd.date_key

order by dc.class_date

In [0]:
--Write your code here

## Question 4 — Student and Class Detail
Build the query in these steps:

**4.1 — Select student columns**
From `dim_student`, select the student name and city, replacing a null city with `Unknown city`.

**4.2 — Join attendance**
Start from `fact_attendance` and join `dim_student` using `student_key`. Select the attendance status.

**4.3 — Add class, batch, and date details**
Join `dim_class`, `dim_batch`, and `dim_date` using their keys. Add class date, day name, batch name, and topic with readable null labels.

**4.4 — Filter valid records**
Keep only rows where attendance status is not null or blank.

**4.5 — Sort the detail**
Order by class date, then student name.

In [0]:
--Write your code here
select 
    ds.student_name,
    coalesce(ds.city,'unknown city') as city,
    dc.class_date,
    coalesce(dd.day_name,'unknown day') as day_name,
    coalesce(db.batch_name,'unknown batch') as batch_name,
    coalesce(dc.topic,'unknown topic') as batch_name,
    fa.attendance_status

from rivadataplatform.dataproduct.dim_student as ds 
left join rivadataplatform.dataproduct.fact_attendance as fa
    on ds.student_key = fa.student_key
left join rivadataplatform.dataproduct.dim_class as dc
    on dc.class_key = fa.class_key
left join rivadataplatform.dataproduct.dim_batch as db
    on db.batch_id = dc.batch_id
left join rivadataplatform.dataproduct.dim_date as dd
        on dd.date_key = fa.date_key

where nullif(fa.attendance_status,'') is not null
order by dc.class_date,ds.student_name

In [0]:
--Write your code here

## Question 5 — Attendance Status Summary
Build the query in these steps:

**5.1 — Select grouping columns**
From attendance, select the batch name and class topic, replacing null values with `No batch` and `Topic not assigned`.

**5.2 — Join dimensions**
Join `dim_class` using `class_key` and `dim_batch` using `batch_key`.

**5.3 — Count attendance**
Count total attendance records and distinct students.

**5.4 — Count each status**
Use conditional aggregation to count `Present`, `Late`, and `Absent` records.

**5.5 — Keep and sort groups**
Keep groups with at least one attendance record and order by batch name, then topic.

In [0]:
--Write your code here
select 
    coalesce(db.batch_name,'No batch') as batch_name,
    coalesce(dc.topic,'Topic not assigned') as topic,
    count(fa.attendance_id) as attendance_count,
    count(distinct fa.student_key) as student_count,
    sum(case when fa.attendance_status = 'present' then 1 else 0 end) as present_flag,
    sum(case when fa.attendance_status = 'absent' then 1 else 0 end) as absent_flag,
    sum(case when fa.attendance_status = 'late' then 1 else 0 end) as late_flag

from rivadataplatform.dataproduct.fact_attendance as fa
    left join rivadataplatform.dataproduct.dim_class as dc
on dc.class_key = fa.class_key
    left join rivadataplatform.dataproduct.dim_batch as db
on db.batch_key = fa.batch_key

group by batch_name, topic
    having attendance_count > 0
order by batch_name, topic

In [0]:
--Write your code here

## Question 6 — Students Requiring Follow-up
Build the query in these steps:

**6.1 — Select student columns**
From `dim_student`, select `student_id` and `student_name`.

**6.2 — Join attendance and dimensions**
Use `LEFT JOIN` to add attendance, class, and batch data through their keys.

**6.3 — Calculate student metrics**
Group by student and calculate total attendance records, issue records, and the most recent class date.

**6.4 — Identify issues**
Count rows whose status is `Late` or `Absent`, and keep only students with at least one such issue.

**6.5 — Sort for follow-up**
Order by issue count descending, then student name.

Return each student once.

In [0]:
--Write your code here
select ds.student_id,ds.student_name,
count(fa.attendance_id) as total_attendance_records,
max(dc.class_date) as recent_class_date,
sum(case when fa.attendance_status in ('late','absent') then 1 else 0 end) as issue_records

from rivadataplatform.dataproduct.dim_student as ds
left join rivadataplatform.dataproduct.fact_attendance as fa
    on ds.student_key = fa.student_key
left join rivadataplatform.dataproduct.dim_class as dc
    on dc.class_key = fa.class_key
left join rivadataplatform.dataproduct.dim_batch as db
    on db.batch_id = dc.batch_id

group by ds.student_id,ds.student_name
having issue_records > 0
order by issue_records DESC, ds.student_name;

In [0]:
--Write your code here